In [ ]:
import requests
import time
import json
import re
import random
from tqdm import tqdm
from google.colab import files

HEADERS = {
    "User-Agent": "MyColabBot/2.0 (inesgoddi@gmail.com)"
}

SESSION = requests.Session()
SESSION.headers.update(HEADERS)

API_SIMPLE = "https://simple.wikipedia.org/w/api.php"
API_EN = "https://en.wikipedia.org/w/api.php"

REQUEST_DELAY = 1.0
MAX_RETRIES = 8

DISCIPLINARY_ROOTS = [
    # Natural sciences
    "Physics", "Chemistry", "Biology", "Astronomy", "Earth science",

    # Formal sciences
    "Mathematics", "Computer science", "Statistics", "Logic",

    # Life sciences
    "Genetics", "Neuroscience", "Ecology", "Microbiology",
    "Zoology", "Botany",

    # Medical sciences
    "Medicine", "Surgery", "Pharmacology", "Epidemiology", "Public health",

    # Engineering
    "Engineering", "Mechanical engineering", "Electrical engineering",
    "Civil engineering", "Chemical engineering", "Aerospace engineering",

    # Social sciences
    "Economics", "Psychology", "Sociology", "Political science",
    "Anthropology", "Human geography",

    # Humanities
    "Philosophy", "History", "Literature", "Linguistics", "Theology",

    # Arts
    "Arts", "Music", "Visual arts", "Performing arts",
    "Film studies", "Architecture",

    # Interdisciplinary
    "Cognitive science", "Environmental science",
    "Data science", "Artificial intelligence", "Systems science"
]

BAD_CATEGORY_PATTERNS = [
    r"^Articles? ", r"^All articles?", r"^Wikipedia ", r"^Pages?",
    r"^CS1 ", r"^Harv ", r"^Use ", r"^Redirects?",
    r"^Disambiguation", r".*errors?$", r".*cleanup.*",
    r".*needing.*", r".*lacking.*", r".*unsourced.*",
    r".*unreferenced.*", r".*broken.*", r".*template.*",
    r".*infobox.*", r".*Wikidata.*", r".*stub.*",
    r".*stubs.*", r".*citation.*", r".*coordinates.*",
    r".*authority control.*", r".*public domain.*",
    r".*incorporating.*", r".*semi-protected.*",
    r".*plot summary.*", r".*bare URLs.*",
    r".*subscription.*", r".*maintenance.*",
    r".*hidden categor.*"
]

BAD_EXACT_CATEGORIES = {
    "Contents", "Lists", "Outlines", "Reference",
    "Comparisons", "Concepts", "Entities",
    "Objects", "Data", "Information"
}

BAD_CATEGORY_REGEX = [re.compile(p, re.IGNORECASE) for p in BAD_CATEGORY_PATTERNS]

DIGIT_PATTERN = re.compile(r"\d")
NON_ALPHA_PATTERN = re.compile(r"[^a-zA-Z\- ]")

BAD_WORDS = {
    "abuse", "abusive", "violence", "violent", "rape", "murder",
    "death", "torture", "prostitution", "porn",
    "sexual", "sex", "cannibal", "cannibalism", "drug",
    "addiction", "gambling", "terrorism", "crime", "criminal",
    "childhood", "children"
}

PEOPLE_PATTERN = re.compile(
    r"\b(bankers?|lawyers?|politicians?|actors?|singers?|players?|"
    r"scientists?|philosophers?|historians?|mathematicians?|"
    r"writers?|authors?|artists?|athletes?|accountants?)\b",
    re.IGNORECASE
)

IST_PATTERN = re.compile(r"\b\w+ist(s)?\b", re.IGNORECASE)
NAME_PATTERN = re.compile(r"^[A-Z][a-z]+ [A-Z][a-z]+$")


def is_disciplinary_category(cat):
    cat = cat.strip()
    cat_lower = cat.lower()

    if cat in BAD_EXACT_CATEGORIES:
        return False

    if any(p.search(cat) for p in BAD_CATEGORY_REGEX):
        return False

    if DIGIT_PATTERN.search(cat):
        return False

    if NON_ALPHA_PATTERN.search(cat):
        return False

    if any(word in cat_lower for word in BAD_WORDS):
        return False

    if PEOPLE_PATTERN.search(cat):
        return False

    if IST_PATTERN.search(cat):
        return False

    if NAME_PATTERN.match(cat):
        return False

    if len(cat.split()) > 1:
        return False
    # strict single-word (letters only, no spaces, no hyphens)
    if not re.fullmatch(r"[A-Za-z]+", cat):
      return False

    return True


def safe_get(api_url, params, max_retries=MAX_RETRIES):
    for attempt in range(max_retries):
        try:
            response = SESSION.get(api_url, params=params, timeout=30)

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    wait = int(retry_after)
                else:
                    wait = min(90, 5 * (2 ** attempt))

                wait += random.uniform(0, 2)
                print(f"429 rate limit. Waiting {wait:.1f}s...")
                time.sleep(wait)
                continue

            response.raise_for_status()

            time.sleep(REQUEST_DELAY + random.uniform(0, 0.5))
            return response.json()

        except requests.exceptions.RequestException as e:
            wait = min(90, 3 * (2 ** attempt)) + random.uniform(0, 2)
            print(f"Request failed: {e}. Waiting {wait:.1f}s...")
            time.sleep(wait)

    return {}


def get_subcategories(category, api_url, depth=5):
    seen = set()
    to_visit = [(category, 0)]
    results = set()

    while to_visit:
        current_cat, current_depth = to_visit.pop()

        if current_cat in seen or current_depth > depth:
            continue

        seen.add(current_cat)
        cmcontinue = None

        while True:
            params = {
                "action": "query",
                "format": "json",
                "list": "categorymembers",
                "cmtitle": f"Category:{current_cat}",
                "cmnamespace": 14,
                "cmlimit": 500
            }

            if cmcontinue:
                params["cmcontinue"] = cmcontinue

            data = safe_get(api_url, params)

            subcats = [
                item["title"].replace("Category:", "")
                for item in data.get("query", {}).get("categorymembers", [])
            ]

            for subcat in subcats:
                if is_disciplinary_category(subcat):
                    results.add(subcat)

                    if current_depth < depth:
                        to_visit.append((subcat, current_depth + 1))

            cmcontinue = data.get("continue", {}).get("cmcontinue")

            if not cmcontinue:
                break

    return results


def fetch_root_subcats(cat):
    print(f"\nProcessing root category: {cat}")

    simple = get_subcategories(cat, API_SIMPLE, depth=7)
    en = get_subcategories(cat, API_EN, depth=7)

    return simple, en


print("Using predefined disciplinary root categories...")

subcategories_simple = set()
subcategories_en = set()

for cat in tqdm(DISCIPLINARY_ROOTS, desc="Getting subcategories"):
    simple, en = fetch_root_subcats(cat)
    subcategories_simple.update(simple)
    subcategories_en.update(en)

common_cats = sorted(
    (subcategories_simple & subcategories_en) | set(DISCIPLINARY_ROOTS)
)

common_cats = [
    cat for cat in common_cats
    if is_disciplinary_category(cat)
]

print(f"\nKept {len(common_cats)} disciplinary categories.")

with open("disciplinary_categories3.json", "w", encoding="utf-8") as f:
    json.dump(common_cats, f, ensure_ascii=False, indent=2)

files.download("disciplinary_categories3.json")

Using predefined disciplinary root categories...


Getting subcategories:   0%|          | 0/48 [00:00<?, ?it/s]


Processing root category: Physics


Getting subcategories:   2%|▏         | 1/48 [02:18<1:48:28, 138.47s/it]


Processing root category: Chemistry


...


Processing root category: Systems science


Getting subcategories: 100%|██████████| 48/48 [2:11:57<00:00, 164.95s/it]


Kept 267 disciplinary categories.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from numpy import char
import requests
import time
import json
import random
import csv
import re
from tqdm import tqdm
import pandas as pd
from google.colab import files

HEADERS = {
    "User-Agent": "MyColabBot/1.0 (inesgoddi@gmail.com)"
}

API_SIMPLE = "https://simple.wikipedia.org/w/api.php"
API_EN = "https://en.wikipedia.org/w/api.php"

DISCIPLINARY_ROOTS = [
    "Science", "Natural sciences",
    "Physics", "Theoretical physics", "Applied physics", "Quantum mechanics",
    "Astrophysics", "Astronomy", "Cosmology", "Particle physics",
    "Biology", "Molecular biology", "Cell biology", "Genetics",
    "Evolutionary biology", "Ecology", "Neuroscience",
    "Microbiology", "Zoology", "Botany",
    "Chemistry", "Organic chemistry", "Inorganic chemistry",
    "Physical chemistry", "Biochemistry", "Analytical chemistry",
    "Earth sciences", "Geology", "Geophysics", "Meteorology",
    "Oceanography", "Environmental science", "Climate science",

    "Mathematics", "Pure mathematics", "Applied mathematics",
    "Algebra", "Geometry", "Topology", "Number theory",
    "Combinatorics", "Mathematical analysis",
    "Statistics", "Probability theory",
    "Logic", "Mathematical logic",
    "Theoretical computer science",

    "Computer science", "Artificial intelligence",
    "Machine learning", "Deep learning",
    "Data science", "Information theory",
    "Information systems", "Software engineering",
    "Computer engineering", "Human–computer interaction",
    "Computer vision", "Natural language processing",
    "Cybersecurity", "Distributed computing",

    "Engineering", "Electrical engineering", "Mechanical engineering",
    "Civil engineering", "Chemical engineering",
    "Aerospace engineering", "Biomedical engineering",
    "Industrial engineering", "Systems engineering",
    "Materials science", "Nanotechnology",

    "Medicine", "Public health", "Epidemiology",
    "Pathology", "Pharmacology", "Immunology",
    "Anatomy", "Physiology",
    "Clinical medicine", "Surgery",
    "Psychiatry", "Nursing", "Dentistry",

    "Social sciences",
    "Economics", "Microeconomics", "Macroeconomics",
    "Political science", "Public policy", "International relations",
    "Sociology", "Social theory",
    "Anthropology", "Cultural anthropology",
    "Human geography", "Economic geography",
    "Demography", "Criminology",
    "Psychology", "Cognitive psychology",
    "Developmental psychology", "Social psychology",
    "Behavioral science",

    "Humanities",
    "Philosophy", "Epistemology", "Metaphysics",
    "Ethics", "Logic (philosophy)", "Philosophy of science",
    "History", "Historiography",
    "Archaeology",
    "Linguistics", "Phonetics", "Syntax", "Semantics",
    "Literary theory", "Literature",
    "Religious studies", "Theology",
    "Classics",

    "Law", "Criminal law", "Civil law",
    "International law", "Constitutional law",
    "Legal theory",

    "Business", "Management", "Organizational behavior",
    "Finance", "Accounting", "Marketing",
    "Operations management",
    "Entrepreneurship",

    "Education", "Pedagogy", "Educational psychology",
    "Curriculum studies", "Instructional design",

    "Cognitive science", "Data analysis",
    "Systems science", "Complex systems",
    "Environmental studies",
    "Sustainability", "Urban studies",
    "Science and technology studies",

    "Communication", "Media studies",
    "Journalism", "Rhetoric",
    "Library science", "Information science",

    "Arts", "Musicology", "Art history",
    "Visual arts", "Performing arts"
]

BAD_CATEGORY_PATTERNS = [
    r"^Articles? ",
    r"^All articles?",
    r"^Wikipedia ",
    r"^Pages? ",
    r"^CS1 ",
    r"^Harv ",
    r"^Use ",
    r"^Redirects? ",
    r"^Disambiguation",
    r".*errors?$",
    r".*cleanup.*",
    r".*needing.*",
    r".*lacking.*",
    r".*unsourced.*",
    r".*unreferenced.*",
    r".*broken.*",
    r".*template.*",
    r".*infobox.*",
    r".*Wikidata.*",
    r".*stub.*",
    r".*stubs.*",
    r".*citation.*",
    r".*coordinates.*",
    r".*authority control.*",
    r".*public domain.*",
    r".*incorporating.*",
    r".*semi-protected.*",
    r".*plot summary.*",
    r".*bare URLs.*",
    r".*subscription.*",
    r".*maintenance.*",
    r".*hidden categor.*"
]

BAD_EXACT_CATEGORIES = {
    "Contents",
    "Lists",
    "Outlines",
    "Reference",
    "Comparisons",
    "Concepts",
    "Entities",
    "Objects",
    "Data",
    "Information"
}



# =========================
# EXTRA FILTERING (NEW)
# =========================

# Compile regex once (faster)
BAD_CATEGORY_REGEX = [re.compile(p, re.IGNORECASE) for p in BAD_CATEGORY_PATTERNS]

DIGIT_PATTERN = re.compile(r"\d")
NON_ALPHA_PATTERN = re.compile(r"[^a-zA-Z\- ]")

BAD_WORDS = {
    "abuse", "abusive", "violence", "violent", "rape", "murder",
    "death", "suicide", "torture", "prostitution", "porn",
    "sexual", "sex", "cannibal", "cannibalism", "drug",
    "addiction", "gambling", "terrorism", "crime", "criminal"
}

PEOPLE_PATTERN = re.compile(
    r"\b(bankers?|lawyers?|politicians?|actors?|singers?|players?|"
    r"scientists?|philosophers?|historians?|mathematicians?)\b",
    re.IGNORECASE
)

IST_PATTERN = re.compile(r"\b\w+ist(s)?\b", re.IGNORECASE)

# Detect likely person names (Firstname Lastname)
NAME_PATTERN = re.compile(r"^[A-Z][a-z]+ [A-Z][a-z]+$")


def is_disciplinary_category(cat):
    cat = cat.strip()
    cat_lower = cat.lower()

    # 1. Exact blacklist
    if cat in BAD_EXACT_CATEGORIES:
        return False

    # 2. Pattern blacklist (Wikipedia junk)
    if any(p.search(cat) for p in BAD_CATEGORY_REGEX):
        return False

    # 3. ❌ Remove digits
    if DIGIT_PATTERN.search(cat):
        return False

    # 4. ❌ Remove weird symbols (optional but strong)
    if NON_ALPHA_PATTERN.search(cat):
        return False

    # 5. ❌ Remove bad semantic words
    if any(word in cat_lower for word in BAD_WORDS):
        return False

    # 6. ❌ Remove professions / social groups
    if PEOPLE_PATTERN.search(cat):
        return False

    # 7. ❌ Remove "-ist" words (bias / ideology)
    if IST_PATTERN.search(cat):
        return False

    # 8. ❌ Remove likely person names
    if NAME_PATTERN.match(cat):
        return False

    if len(cat.split()) > 3:
      return False

    return True

def safe_get(api_url, params, max_retries=3):
    for attempt in range(max_retries):
        try:
            r = requests.get(api_url, headers=HEADERS, params=params, timeout=30)
            r.raise_for_status()
            return r.json()
        except Exception as e:
            print(f"Request failed: {e}. Retry {attempt + 1}/{max_retries}")
            time.sleep(1)

    return {}

def get_subcategories(category, api_url, depth=5):
    seen = set()
    to_visit = [(category, 0)]
    results = set()

    while to_visit:
        current_cat, current_depth = to_visit.pop()

        if current_cat in seen or current_depth > depth:
            continue

        seen.add(current_cat)
        cmcontinue = None

        while True:
            params = {
                "action": "query",
                "format": "json",
                "list": "categorymembers",
                "cmtitle": f"Category:{current_cat}",
                "cmnamespace": 14,
                "cmlimit": "max"
            }

            if cmcontinue:
                params["cmcontinue"] = cmcontinue

            data = safe_get(api_url, params)

            subcats = [
                item["title"].replace("Category:", "")
                for item in data.get("query", {}).get("categorymembers", [])
            ]

            for subcat in subcats:
                if is_disciplinary_category(subcat):
                    results.add(subcat)
                    to_visit.append((subcat, current_depth + 1))

            cmcontinue = data.get("continue", {}).get("cmcontinue")
            if not cmcontinue:
                break

            time.sleep(0.1)

    return results

def get_category_articles(category, api_url, limit=100):
    titles = []
    cmcontinue = None

    while len(titles) < limit:
        params = {
            "action": "query",
            "format": "json",
            "list": "categorymembers",
            "cmtitle": f"Category:{category}",
            "cmnamespace": 0,
            "cmlimit": "max"
        }

        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        data = safe_get(api_url, params)

        titles.extend([
            page["title"]
            for page in data.get("query", {}).get("categorymembers", [])
        ])

        cmcontinue = data.get("continue", {}).get("cmcontinue")
        if not cmcontinue:
            break

        time.sleep(0.1)

    return titles[:limit]

def filter_existing_titles_in_en(titles):
    existing = []
    batch_size = 50

    for i in range(0, len(titles), batch_size):
        batch = titles[i:i + batch_size]

        params = {
            "action": "query",
            "format": "json",
            "titles": "|".join(batch),
            "redirects": 1
        }

        data = safe_get(API_EN, params)

        for page in data.get("query", {}).get("pages", {}).values():
            if "missing" not in page:
                existing.append(page["title"])

        time.sleep(0.1)

    return existing

def get_page_extract(title, api_url):
    params = {
        "action": "query",
        "format": "json",
        "prop": "extracts",
        "explaintext": True,
        "titles": title,
        "redirects": 1
    }

    data = safe_get(api_url, params)

    pages = data.get("query", {}).get("pages", {})

    for page in pages.values():
        if "missing" in page:
            return None
        return page.get("extract", "")

    return None

def is_valid_text(text, min_words=100):
    if not text:
        return False

    words = text.split()

    if len(words) < min_words:
        return False

    return True
'''
# =========================
# 1. GET DISCIPLINARY CATEGORIES
# =========================

print("Using predefined disciplinary root categories...")

subcategories_simple = set()
subcategories_en = set()

for cat in tqdm(DISCIPLINARY_ROOTS, desc="Getting subcategories"):
    subcategories_simple.update(get_subcategories(cat, API_SIMPLE, depth=2))
    subcategories_en.update(get_subcategories(cat, API_EN, depth=2))

common_cats = sorted(
    (subcategories_simple & subcategories_en) | set(DISCIPLINARY_ROOTS)
)

common_cats = [
    cat for cat in common_cats
    if is_disciplinary_category(cat)
]

print(f"Kept {len(common_cats)} disciplinary categories.")

with open("disciplinary_categories2.json", "w", encoding="utf-8") as f:
    json.dump(common_cats, f, ensure_ascii=False, indent=2)

files.download("disciplinary_categories2.json")


# =========================
# 2. COLLECT ARTICLE TITLES
# =========================

matched_pairs = []

MAX_TOTAL_PAIRS = 10000
MAX_ARTICLES_PER_CATEGORY = 100

for cat in tqdm(common_cats, desc="Collecting articles"):
    if len(cat.split()) >1 or any(char.isdigit() for char in cat):
        continue

    print(f"Collecting articles for category: {cat}")
    titles_simple = get_category_articles(
        cat,
        API_SIMPLE,
        limit=MAX_ARTICLES_PER_CATEGORY * 2
    )

    if not titles_simple:
        continue

    existing_in_en = filter_existing_titles_in_en(titles_simple)

    count_for_cat = 0

    for title in existing_in_en:
        matched_pairs.append({
            "category": cat,
            "title": title
        })

        count_for_cat += 1

        if count_for_cat >= MAX_ARTICLES_PER_CATEGORY:
            break

        if len(matched_pairs) >= MAX_TOTAL_PAIRS:
            break

    if len(matched_pairs) >= MAX_TOTAL_PAIRS:
        break

with open("category_title_pairs_disciplinary.json", "w", encoding="utf-8") as f:
    json.dump(matched_pairs, f, ensure_ascii=False, indent=2)

print(f"Collected {len(matched_pairs)} category-title pairs.")
print(matched_pairs)
'''
# =========================
# 3. EXTRACT SIMPLE + EN TEXTS
# =========================
'''
results = []

for pair in tqdm(matched_pairs, desc="Extracting texts"):
    cat = pair["category"]
    title = pair["title"]

    simple_text = get_page_extract(title, API_SIMPLE)
    en_text = get_page_extract(title, API_EN)

    if not is_valid_text(simple_text, min_words=100):
        continue

    if not is_valid_text(en_text, min_words=100):
        continue

    results.append({
        "category": cat,
        "title": title,
        "simple_text": simple_text,
        "en_text": en_text
    })

    time.sleep(0.1)

with open("category_title_texts_disciplinary.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Extracted valid text pairs for {len(results)} articles.")

# =========================
# 4. CREATE PAIRWISE CSV
# =========================

output_rows = []

for entry in results:
    simple_text = entry["simple_text"].strip()
    en_text = entry["en_text"].strip()
    category = entry["category"].strip()
    title = entry["title"].strip()

    label = random.randint(0, 1)

    if label == 1:
        sentence_1 = en_text
        sentence_2 = simple_text
    else:
        sentence_1 = simple_text
        sentence_2 = en_text

    output_rows.append({
        "sentence_1": sentence_1,
        "sentence_2": sentence_2,
        "label": label,
        "category": category,
        "title": title
    })

# Balance labels
label_0 = [row for row in output_rows if row["label"] == 0]
label_1 = [row for row in output_rows if row["label"] == 1]

min_len = min(len(label_0), len(label_1))

balanced_rows = label_0[:min_len] + label_1[:min_len]
random.shuffle(balanced_rows)

with open("output_with_disciplinary_category.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = [
        "sentence_1",
        "sentence_2",
        "label",
        "category",
        "title"
    ]

    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(balanced_rows)

print(f"Final balanced dataset size: {len(balanced_rows)}")
print("Saved: output_with_disciplinary_category.csv")
'''

'\nresults = []\n\nfor pair in tqdm(matched_pairs, desc="Extracting texts"):\n    cat = pair["category"]\n    title = pair["title"]\n\n    simple_text = get_page_extract(title, API_SIMPLE)\n    en_text = get_page_extract(title, API_EN)\n\n    if not is_valid_text(simple_text, min_words=100):\n        continue\n\n    if not is_valid_text(en_text, min_words=100):\n        continue\n\n    results.append({\n        "category": cat,\n        "title": title,\n        "simple_text": simple_text,\n        "en_text": en_text\n    })\n\n    time.sleep(0.1)\n\nwith open("category_title_texts_disciplinary.json", "w", encoding="utf-8") as f:\n    json.dump(results, f, ensure_ascii=False, indent=2)\n\nprint(f"Extracted valid text pairs for {len(results)} articles.")\n\n# =========================\n# 4. CREATE PAIRWISE CSV\n# =========================\n\noutput_rows = []\n\nfor entry in results:\n    simple_text = entry["simple_text"].strip()\n    en_text = entry["en_text"].strip()\n    category = en

In [ ]:
len(matched_pairs)

10000

In [ ]:
import re

BAD_KEYWORDS = {
    "prostitution", "porn", "sexual", "abuse", "violence",
    "rape", "cannibal", "drug", "addiction", "gambling",
    "suicide", "murder", "death", "torture"
}

BAD_CATEGORIES = {
    "abuse", "addiction", "adultery", "abortion"
}

IST_PATTERN = re.compile(r"\b\w+ist(s)?\b", re.IGNORECASE)
PEOPLE_PATTERN = re.compile(r"\b(bankers?|lawyers?|politicians?)\b", re.IGNORECASE)

def is_clean(pair):
    cat = pair["category"].lower()
    title = pair["title"].lower()

    text = f"{cat} {title}"

    # 0. ❗ Keep ONLY single-word titles
    # (removes "ASEAN Declaration", "Physical abuse", etc. from your file :contentReference[oaicite:0]{index=0})
    if len(title.split()) != 1:
        return False

    # 1. Remove bad categories
    if cat in BAD_CATEGORIES:
        return False

    # 2. Remove unwanted keywords
    if any(keyword in text for keyword in BAD_KEYWORDS):
        return False

    # 3. Remove professions / groups
    if PEOPLE_PATTERN.search(text):
        return False

    # 4. Remove words ending with -ist
    if IST_PATTERN.search(text):
        return False

    return True

In [ ]:
clean_pairs = []

for pair in tqdm(matched_pairs, desc="Filtering"):
    if is_clean(pair):
        clean_pairs.append(pair)

Filtering: 100%|██████████| 10000/10000 [00:00<00:00, 446868.10it/s]


In [ ]:
import json

with open("disciplinary_categories3.json", "r") as file:
    common_cats= json.load(file)

print(len(common_cats))

267


In [ ]:
common_cats
n = [
    "Anniversaries",
    "Architects",
    "Astronauts",
    "Astronomers",
    "Beekeepers",
    "Birthdays",
    "Choreographers",
    "Dancers",
    "Engineers",
    "Firefighters",
    "Islamophobia",
    "Plumbers",
    "Statisticians",
    "Surgeons",
    "Woodworkers",
    "Zoos"
]


result = [x for x in common_cats if x not in n]

print(len(result))

257


In [ ]:
from numpy import char
import requests
import time
import json
import random
import csv
import re
from tqdm import tqdm
import pandas as pd
from google.colab import files

HEADERS = {
    "User-Agent": "MyColabBot/1.0 (inesgoddi@gmail.com)"
}

API_SIMPLE = "https://simple.wikipedia.org/w/api.php"
API_EN = "https://en.wikipedia.org/w/api.php"

# =========================
# 2. COLLECT ARTICLE TITLES
# =========================

matched_pairs = []

MAX_TOTAL_PAIRS = 10000
MAX_ARTICLES_PER_CATEGORY = 100

for cat in tqdm(result, desc="Collecting articles"):
    if len(cat.split()) >1 or any(char.isdigit() for char in cat):
        continue

    print(f"Collecting articles for category: {cat}")
    titles_simple = get_category_articles(
        cat,
        API_SIMPLE,
        limit=MAX_ARTICLES_PER_CATEGORY * 2
    )

    if not titles_simple:
        continue

    existing_in_en = filter_existing_titles_in_en(titles_simple)

    count_for_cat = 0

    for title in existing_in_en:
        matched_pairs.append({
            "category": cat,
            "title": title
        })

        count_for_cat += 1

        if count_for_cat >= MAX_ARTICLES_PER_CATEGORY:
            break

        if len(matched_pairs) >= MAX_TOTAL_PAIRS:
            break

    if len(matched_pairs) >= MAX_TOTAL_PAIRS:
        break

with open("category_title_pairs_disciplinary3.json", "w", encoding="utf-8") as f:
    json.dump(matched_pairs, f, ensure_ascii=False, indent=2)

print(f"Collected {len(matched_pairs)} category-title pairs.")
print(matched_pairs)

Collected 6342 category-title pairs.
[{'category': 'Adhesives', 'title': 'Adhesion'}, {'category': 'Adhesives', 'title': 'Adhesive'}, {'category': 'Adhesives', 'title': 'Binder (material)'}, {'category': 'Adhesives', 'title': 'Epoxy'}, {'category': 'Adhesives', 'title': 'Sticker'}, {'category': 'Adhesives', 'title': 'Cyanoacrylate'}, {'category': 'Adhesives', 'title': 'Thickening agent'}, {'category': 'Aircraft', 'title': '3Xtrim 3X55 Trener'}, {'category': 'Aircraft', 'title': 'A2 CZ Ellipse Spirit'}, {'category': 'Aircraft', 'title': 'ATR 72'}, {'category': 'Aircraft', 'title': 'Acrolite'}, {'category': 'Aircraft', 'title': 'AeroAndina MXP-150 Kimbaya'}, {'category': 'Aircraft', 'title': 'AeroAndina MXP-158 Embera'}, {'category': 'Aircraft', 'title': 'Aero Ae-45'}, {'category': 'Aircraft', 'title': 'Aircraft'}, {'category': 'Aircraft', 'title': 'Airliner'}, {'category': 'Aircraft', 'title': 'Airplane'}, {'category': 'Aircraft', 'title': 'American Airlines fleet'}, {'category': 'Aircr

In [ ]:
# =========================
# 3. EXTRACT SIMPLE + EN TEXTS
# =========================

results = []

for pair in tqdm(matched_pairs, desc="Extracting texts"):
    cat = pair["category"]
    title = pair["title"]

    simple_text = get_page_extract(title, API_SIMPLE)
    en_text = get_page_extract(title, API_EN)


    results.append({
        "category": cat,
        "title": title,
        "simple_text": simple_text,
        "en_text": en_text
    })

    time.sleep(0.1)

with open("category_title_texts_disciplinary3.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Extracted valid text pairs for {len(results)} articles.")

# =========================
# 4. CREATE PAIRWISE CSV
# =========================

output_rows = []

for entry in results:
    if entry is None or entry.get("simple_text") is None or entry.get("en_text") is None:
      continue
    simple_text = entry["simple_text"].strip()
    en_text = entry["en_text"].strip()
    category = entry["category"].strip()
    title = entry["title"].strip()

    label = random.randint(0, 1)

    if label == 1:
        sentence_1 = en_text
        sentence_2 = simple_text
    else:
        sentence_1 = simple_text
        sentence_2 = en_text

    output_rows.append({
        "sentence_1": sentence_1,
        "sentence_2": sentence_2,
        "label": label,
        "category": category,
        "title": title
    })

# Balance labels
label_0 = [row for row in output_rows if row["label"] == 0]
label_1 = [row for row in output_rows if row["label"] == 1]

min_len = min(len(label_0), len(label_1))

balanced_rows = label_0[:min_len] + label_1[:min_len]
random.shuffle(balanced_rows)

with open("output_with_disciplinary_category.csv", "w", newline="", encoding="utf-8") as csvfile:
    fieldnames = [
        "sentence_1",
        "sentence_2",
        "label",
        "category",
        "title"
    ]

    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(balanced_rows)

print(f"Final balanced dataset size: {len(balanced_rows)}")
print("Saved: output_with_disciplinary_category.csv")

Extracting texts: 100%|██████████| 6342/6342 [1:15:15<00:00,  1.40it/s]


Extracted valid text pairs for 6342 articles.
Final balanced dataset size: 5808
Saved: output_with_disciplinary_category.csv


In [ ]:
count = 0
for i, entry in enumerate(results):
    if entry is None or entry.get("simple_text") is None:
        count += 1
print(count)

483


In [ ]:
import pandas as pd

df = pd.read_csv("output_with_disciplinary_category.csv")
print(len(df))
print(df.head())

5808
                                          sentence_1  \
0  Archosauromorpha is a clade of diapsid reptile...   
1  Detective Conan (名探偵コナン, Meitantei Konan), als...   
2  Consonant mutation is change in a consonant in...   
3  Xenophobia (from Ancient Greek  ξένος (xénos) ...   
4  The meat ant (Iridomyrmex purpureus), also cal...   

                                          sentence_2  label     category  \
0  Archosauromorpha (Greek for "ruling lizard for...      0     Reptiles   
1  Case Closed, also officially known as Detectiv...      0        Manga   
2  Consonant mutation is a feature in languages w...      1  Linguistics   
3  Xenophobia is the fear or dislike of strangers...      1   Xenophobia   
4  The meat ant (Iridomyrmex purpureus), also kno...      0         Ants   

                title  
0    Archosauromorpha  
1         Case Closed  
2  Consonant mutation  
3          Xenophobia  
4            Meat ant  


In [ ]:
df

,sentence_1,sentence_2,label,category,title
0,Archosauromorpha is a clade of diapsid reptile...,"Archosauromorpha (Greek for ""ruling lizard for...",0,Reptiles,Archosauromorpha
1,"Detective Conan (名探偵コナン, Meitantei Konan), als...","Case Closed, also officially known as Detectiv...",0,Manga,Case Closed
2,Consonant mutation is change in a consonant in...,Consonant mutation is a feature in languages w...,1,Linguistics,Consonant mutation
3,Xenophobia (from Ancient Greek ξένος (xénos) ...,Xenophobia is the fear or dislike of strangers...,1,Xenophobia,Xenophobia
4,"The meat ant (Iridomyrmex purpureus), also cal...","The meat ant (Iridomyrmex purpureus), also kno...",0,Ants,Meat ant
...,...,...,...,...,...
5803,The mole (symbol: mol) is the SI unit used to ...,The mole (symbol mol) is a unit of measurement...,0,Chemistry,Mole (unit)
5804,A siphon is a long tube-like structure that is...,A siphon is an anatomical structure which is p...,0,Molluscs,Siphon (mollusc)
5805,Shogun Warriors may refer to:\n\nShogun Warrio...,Shogun Warriors was a line of robot toyline re...,1,Robots,Shogun Warriors
5806,Discovery is the act of detecting something ne...,Discovery is the act of detecting something ne...,1,Cognition,Discovery (observation)


In [ ]:
pip install pylatexenc


In [ ]:
# ---------------------------
# Install ftfy if needed
# ---------------------------
try:
    import ftfy
except ImportError:
    !pip install ftfy
    import ftfy

# ---------------------------
# Imports
# ---------------------------
import pandas as pd
from bs4 import BeautifulSoup
import re
from google.colab import drive
from pylatexenc.latex2text import LatexNodes2Text
'''
# ---------------------------
# Mount Drive
# ---------------------------
drive.mount('/content/drive')

# ---------------------------
# 1️⃣ Load CSV safely
# ---------------------------
file_path = '/content/drive/MyDrive/output_with_disciplinary_category.csv'
'''
import pandas as pd
import csv

file_path = "output_with_disciplinary_category.csv"

try:
    df1 = pd.read_csv(file_path, encoding="utf-8")
except UnicodeDecodeError:
    df1 = pd.read_csv(file_path, encoding="latin1")

print("CSV loaded. Rows:", len(df1))

# ---------------------------
# 2️⃣ Preserve original content
# ---------------------------
df1['sentence_1_orig'] = df1['sentence_1']
df1['sentence_2_orig'] = df1['sentence_2']

# ---------------------------
# 3️⃣ Cleaning function with reason tracking
# ---------------------------
def clean_text_with_reason(text):
    if pd.isna(text):
        return "", "original_nan"

    original = str(text)

    # Fix encoding
    cleaned = ftfy.fix_text(original)

    # Remove HTML
    cleaned = BeautifulSoup(cleaned, "html.parser").get_text()

    converter = LatexNodes2Text(strict_latex=False)
    cleaned = converter.latex_to_text(cleaned)

    # Step 2: Remove leftover {\displaystyle ...} blocks
    cleaned = re.sub(r'\{\\displaystyle\s+.*?\}', '', cleaned, flags=re.DOTALL)

    # Step 3: Remove zero-width and formatting artifacts
    cleaned = cleaned.replace('\u2061', '')  # invisible function-application char

    # Remove wiki headers
    cleaned = re.sub(r'==.*?==', '', cleaned)

    # Normalize whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    # Content destroyed by cleaning
    if original.strip() != "" and cleaned == "":
        return "", "content_removed_by_cleaning"

    return cleaned, ""

# ---------------------------
# 4️⃣ Apply cleaning with reasons
# ---------------------------
df1[['sentence_1_clean', 'sentence_1_reason']] = (
    df1['sentence_1'].apply(lambda x: pd.Series(clean_text_with_reason(x)))
)

df1[['sentence_2_clean', 'sentence_2_reason']] = (
    df1['sentence_2'].apply(lambda x: pd.Series(clean_text_with_reason(x)))
)

# ---------------------------
# 5️⃣ Convert destroyed content to NaN
# ---------------------------
df1.loc[df1['sentence_1_reason'] == "content_removed_by_cleaning", 'sentence_1_clean'] = pd.NA
df1.loc[df1['sentence_2_reason'] == "content_removed_by_cleaning", 'sentence_2_clean'] = pd.NA

# ---------------------------
# 6️⃣ Report destroyed rows
# ---------------------------
for col in ['sentence_1', 'sentence_2']:
    reason_col = f"{col}_reason"
    clean_col = f"{col}_clean"
    orig_col = f"{col}_orig"

    count = (df1[reason_col] == "content_removed_by_cleaning").sum()

    if count > 0:
        print(f"\n⚠️ {count} rows in {col} became NaN due to cleaning")
        print(df1.loc[df1[reason_col] == "content_removed_by_cleaning",
                     [orig_col, clean_col]].head())
    else:
        print(f"✅ No content destroyed in {col}")

# ---------------------------
# 7️⃣ Optional: check leftover markup
# ---------------------------
html_like = re.compile(r'<[^>]+>|==.*==|\$.*\$')

suspicious_rows = df1[
    df1['sentence_1_clean'].astype(str).str.contains(html_like, na=False) |
    df1['sentence_2_clean'].astype(str).str.contains(html_like, na=False)
]

if not suspicious_rows.empty:
    print(f"\n⚠️ Suspicious markup remains in {len(suspicious_rows)} rows")
else:
    print("\n✅ No leftover markup detected")

# ---------------------------
# 8️⃣ Save cleaned dataframe
# ---------------------------

# Drop rows where sentence_1_clean or sentence_2_clean is NaN
df1 = df1.dropna(subset=['sentence_1_clean', 'sentence_2_clean'])

# Optional: reset the index
df1 = df1.reset_index(drop=True)

print(f"✅ Remaining rows: {len(df1)}")

df1['sentence_1'] = df1['sentence_1_clean']
df1['sentence_2'] = df1['sentence_2_clean']
df1 = df1[['sentence_1', 'sentence_2', 'label','category','title']]

output_path = 'output_cleaned_title.csv'
df1.to_csv(output_path, index=False)
print(f"\n💾 Cleaned CSV saved to: {output_path}")


CSV loaded. Rows: 5808


Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.


✅ No content destroyed in sentence_1
✅ No content destroyed in sentence_2

⚠️ Suspicious markup remains in 112 rows
✅ Remaining rows: 5808

💾 Cleaned CSV saved to: output_cleaned_title.csv


In [ ]:
df1

,sentence_1,sentence_2,label,category
0,Archosauromorpha is a clade of diapsid reptile...,"Archosauromorpha (Greek for ""ruling lizard for...",0,Reptiles
1,"Detective Conan (名探偵コナン, Meitantei Konan), als...","Case Closed, also officially known as Detectiv...",0,Manga
2,Consonant mutation is change in a consonant in...,Consonant mutation is a feature in languages w...,1,Linguistics
3,Xenophobia (from Ancient Greek ξένος (xénos) '...,Xenophobia is the fear or dislike of strangers...,1,Xenophobia
4,"The meat ant (Iridomyrmex purpureus), also cal...","The meat ant (Iridomyrmex purpureus), also kno...",0,Ants
...,...,...,...,...
5803,The mole (symbol: mol) is the SI unit used to ...,The mole (symbol mol) is a unit of measurement...,0,Chemistry
5804,A siphon is a long tube-like structure that is...,A siphon is an anatomical structure which is p...,0,Molluscs
5805,Shogun Warriors may refer to: Shogun Warriors ...,Shogun Warriors was a line of robot toyline re...,1,Robots
5806,Discovery is the act of detecting something ne...,Discovery is the act of detecting something ne...,1,Cognition


In [ ]:
from google.colab import files
files.download('output_cleaned_title.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

df = pd.read_csv("output_cleaned_title.csv")
print(len(df))
print(df.head())

5808
                                          sentence_1  \
0  Archosauromorpha is a clade of diapsid reptile...   
1  Detective Conan (名探偵コナン, Meitantei Konan), als...   
2  Consonant mutation is change in a consonant in...   
3  Xenophobia (from Ancient Greek ξένος (xénos) '...   
4  The meat ant (Iridomyrmex purpureus), also cal...   

                                          sentence_2  label     category  \
0  Archosauromorpha (Greek for "ruling lizard for...      0     Reptiles   
1  Case Closed, also officially known as Detectiv...      0        Manga   
2  Consonant mutation is a feature in languages w...      1  Linguistics   
3  Xenophobia is the fear or dislike of strangers...      1   Xenophobia   
4  The meat ant (Iridomyrmex purpureus), also kno...      0         Ants   

                title  
0    Archosauromorpha  
1         Case Closed  
2  Consonant mutation  
3          Xenophobia  
4            Meat ant  
